
# ARC-v0.15 — Deployable Boundary Prediction and Signed Harmful-Amplification Audit

This notebook addresses two acceptance-critical validity questions for the SIGIR Full Paper:

1. **Deployability:** can amplification risk be predicted without qrels, without SQ8-side features, and without first executing the higher-fidelity retriever?
2. **Directionality:** when the original absolute utility-gap trajectory amplifies, is that usually *harmful* to the lower-fidelity retriever, or merely a growing disagreement between the two trajectories?

## Scope

This notebook is a **post-hoc validity audit** over the already-sealed ARC-v0.13 FEVER artifacts.

It:

- reads the frozen ARC-v0.13 FIT and untouched validation checkpoints;
- does **not** rerun 5M-document retrieval;
- does **not** access test data;
- keeps the frozen regime threshold \(|H3| = 0.002\);
- does **not** revive the degenerate q75 target;
- separates diagnostic analysis from deployable inference.

## Primary outputs

### Part A — Deployable selector

Allowed features:

- PQ32-only score entropy
- PQ32-only top-1 vs top-10 margin
- policy parameters \(\alpha, k, \text{method}, T\)

Forbidden features:

- qrels-derived utility
- SQ8 scores
- SQ8 candidates
- PQ32↔SQ8 divergence
- any feature requiring the higher-fidelity retriever

Primary validation metrics:

- ROC-AUC
- PR-AUC
- query-cluster bootstrap confidence intervals
- top-risk enrichment
- 25% risk routing vs 25% random routing

### Part B — Signed harmful-amplification audit

For each query-policy trajectory:

\[
G_t = nDCG_{SQ8}(t) - nDCG_{PQ32}(t)
\]

We compare:

\[
H3_{abs} = slope(|G_t|)
\]

with:

\[
H3_{signed} = slope(G_t)
\]

and classify original absolute-amplification events into:

- harmful amplification
- beneficial divergence
- unresolved / tied


In [ ]:

# ============================================================
# Cell 1 — Robust preflight / source integrity
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import warnings
import gc

import numpy as np
import pandas as pd

from google.colab import drive

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 20260816
EPS = 0.002
BOOTSTRAP_REPS = 1000
RANDOM_BASELINE_REPS = 2000

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), "Google Drive mount failed"

ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"
assert ARC_ROOT.is_dir(), ARC_ROOT

V013_ROOT = ARC_ROOT / "fever-boundary-external-replication-v013"
assert V013_ROOT.is_dir(), V013_ROOT

PREFERRED_V013_RUN = V013_ROOT / "20260817-140640"

def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

def inspect_v013_run(path):
    return {
        "path": path,
        "fit_count": len(list(path.glob("fit-*.parquet"))) if path.is_dir() else 0,
        "val_count": len(list(path.glob("validation-*.parquet"))) if path.is_dir() else 0,
        "protocol": (path / "v013_fever_boundary_protocol.json").is_file(),
        "validation_report": (path / "v013_validation_continuation_report.json").is_file(),
        "validation_sha": (path / "V013_VALIDATION_REPORT_SHA256.txt").is_file(),
    }

def is_complete_v013_run(path):
    i = inspect_v013_run(path)
    return (
        path.is_dir()
        and i["fit_count"] == 44
        and i["val_count"] == 44
        and i["protocol"]
        and i["validation_report"]
        and i["validation_sha"]
    )

if is_complete_v013_run(PREFERRED_V013_RUN):
    V013_RUN = PREFERRED_V013_RUN
else:
    candidates = sorted(
        [p for p in V013_ROOT.iterdir() if p.is_dir()],
        reverse=True,
    )
    valid = [p for p in candidates if is_complete_v013_run(p)]
    assert valid, "No complete ARC-v0.13 run found"
    V013_RUN = valid[0]

PROTOCOL_PATH = V013_RUN / "v013_fever_boundary_protocol.json"
VALIDATION_REPORT_PATH = V013_RUN / "v013_validation_continuation_report.json"
VALIDATION_SHA_PATH = V013_RUN / "V013_VALIDATION_REPORT_SHA256.txt"

fit_files = sorted(V013_RUN.glob("fit-*.parquet"))
val_files = sorted(V013_RUN.glob("validation-*.parquet"))

assert len(fit_files) == 44
assert len(val_files) == 44

protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
validation_report = json.loads(
    VALIDATION_REPORT_PATH.read_text(encoding="utf-8")
)

assert protocol["status"] == "FEVER_BOUNDARY_EXTERNAL_REPLICATION_SEALED_BEFORE_SWEEP"
assert validation_report["status"] == "ARC_V013_FEVER_VALIDATION_CONTINUATION_COMPLETE"
assert protocol["test_access_allowed"] is False
assert validation_report["test_accessed"] is False
assert np.isclose(float(protocol["regime_threshold_abs_slope"]), EPS)

expected_sha = VALIDATION_SHA_PATH.read_text(encoding="utf-8").strip().split()[0]
actual_sha = sha256_file(VALIDATION_REPORT_PATH)
assert expected_sha == actual_sha, (expected_sha, actual_sha)

V015_ROOT = ARC_ROOT / "deployable-boundary-signed-harm-audit-v015"
V015_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
V015_OUT = V015_ROOT / RUN_ID
V015_OUT.mkdir(parents=True, exist_ok=False)

print("Drive:", DRIVE_ROOT)
print("Source:", V013_RUN)
print("FIT checkpoints:", len(fit_files))
print("VAL checkpoints:", len(val_files))
print("Frozen EPS:", EPS)
print("Output:", V015_OUT)
print("ARC-v0.15 CELL 1 PREFLIGHT — PASS")


In [ ]:

# ============================================================
# Cell 2 — Load trajectory checkpoints
# ============================================================

fit_frames = []
for i, p in enumerate(fit_files, 1):
    df = pd.read_parquet(p)
    fit_frames.append(df)
    if i in [1, 10, 20, 30, 40, 44]:
        print(f"[FIT {i:02d}/44]", p.name, df.shape)

val_frames = []
for i, p in enumerate(val_files, 1):
    df = pd.read_parquet(p)
    val_frames.append(df)
    if i in [1, 10, 20, 30, 40, 44]:
        print(f"[VAL {i:02d}/44]", p.name, df.shape)

fit_traj = pd.concat(fit_frames, ignore_index=True)
val_traj = pd.concat(val_frames, ignore_index=True)

del fit_frames, val_frames
gc.collect()

assert fit_traj["config_key"].nunique() == 44
assert val_traj["config_key"].nunique() == 44
assert fit_traj["query_id"].nunique() == 3350
assert val_traj["query_id"].nunique() == 3316

print("FIT trajectory:", fit_traj.shape)
print("VAL trajectory:", val_traj.shape)
print("TRAJECTORY LOAD — PASS")


In [ ]:

# ============================================================
# Cell 3 — Reconstruct slope tables
# ============================================================

GROUP_COLS = [
    "query_id",
    "low",
    "high",
    "method",
    "alpha",
    "k",
    "temperature",
    "config_key",
]

def slopes_from_trajectory(df):
    rows = []

    for keys, g in df.groupby(GROUP_COLS, dropna=False, sort=False):
        g = g.sort_values("iteration")
        x = g["iteration"].to_numpy(np.float64)

        row = dict(zip(GROUP_COLS, keys))

        for src, dst in [
            ("query_divergence", "H1_slope"),
            ("candidate_increment", "H2_slope"),
            ("abs_utility_gap", "H3_abs_slope"),
        ]:
            y = g[src].to_numpy(np.float64)
            row[dst] = float(np.polyfit(x, y, 1)[0])

        rows.append(row)

    return pd.DataFrame(rows)

fit_slopes = slopes_from_trajectory(fit_traj)
val_slopes = slopes_from_trajectory(val_traj)

assert len(fit_slopes) == 3350 * 44
assert len(val_slopes) == 3316 * 44

for df in [fit_slopes, val_slopes]:
    df["is_amplifying_abs"] = (df["H3_abs_slope"] > EPS).astype(int)
    df["is_reversal_abs"] = (df["H3_abs_slope"] < -EPS).astype(int)
    df["is_stable_abs"] = (
        (df["H3_abs_slope"] >= -EPS)
        & (df["H3_abs_slope"] <= EPS)
    ).astype(int)

print("FIT slope rows:", fit_slopes.shape)
print("VAL slope rows:", val_slopes.shape)
print("SLOPE RECONSTRUCTION — PASS")



## Part A — Deployable selector

The selector is intentionally restricted to features available **without qrels and without SQ8**.

The original ARC-v0.13 trajectory checkpoints do not contain raw PQ32 score entropy/margins. Therefore this notebook first searches for a previously persisted baseline-feature artifact under the ARC workspace. If no suitable PQ32-only feature artifact is available, the notebook **fails closed** rather than silently using non-deployable features.

Expected required columns:

- `query_id`
- `pq32_entropy20`
- `pq32_margin1_10`

Policy variables are joined from the trajectory slope table.


In [ ]:

# ============================================================
# Cell 4 — Locate persisted PQ32-only baseline features
# ============================================================

required_feature_cols = {
    "query_id",
    "pq32_entropy20",
    "pq32_margin1_10",
}

candidate_feature_files = []

search_roots = [
    ARC_ROOT,
    DRIVE_ROOT / "rag-pq-checkpoints",
]

for root in search_roots:
    if not root.is_dir():
        continue

    for p in root.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in {".parquet", ".csv"}:
            continue
        name = p.name.lower()
        if any(token in name for token in [
            "boundary",
            "feature",
            "validation_query_config_rows",
            "fit_query_config",
        ]):
            candidate_feature_files.append(p)

print("Candidate feature files:", len(candidate_feature_files))

feature_candidates = []

for p in candidate_feature_files:
    try:
        if p.suffix.lower() == ".parquet":
            tmp = pd.read_parquet(p)
        else:
            tmp = pd.read_csv(p)

        if required_feature_cols.issubset(tmp.columns):
            n_q = tmp["query_id"].nunique()
            feature_candidates.append((p, n_q, tmp.shape))
            print("MATCH:", p, "queries=", n_q, "shape=", tmp.shape)

    except Exception:
        pass

assert feature_candidates, (
    "No persisted artifact with query_id + pq32_entropy20 + "
    "pq32_margin1_10 was found. "
    "Do not substitute SQ8/qrels-dependent features. "
    "Locate/export a sealed PQ32-only baseline feature table first."
)

# Prefer the candidate covering all FEVER queries if one exists.
feature_candidates = sorted(
    feature_candidates,
    key=lambda x: (x[1], str(x[0])),
    reverse=True,
)

FEATURE_SOURCE = feature_candidates[0][0]

print()
print("Selected feature source:", FEATURE_SOURCE)
print("FEATURE SOURCE RESOLUTION — PASS")


In [ ]:

# ============================================================
# Cell 5 — Build deployable feature tables
# ============================================================

if FEATURE_SOURCE.suffix.lower() == ".parquet":
    baseline_features_raw = pd.read_parquet(FEATURE_SOURCE)
else:
    baseline_features_raw = pd.read_csv(FEATURE_SOURCE)

baseline_features = (
    baseline_features_raw[
        ["query_id", "pq32_entropy20", "pq32_margin1_10"]
    ]
    .drop_duplicates("query_id")
    .copy()
)

def add_policy_features(df):
    out = df.copy()
    out["is_softmax"] = (out["method"] == "softmax").astype(float)
    out["temperature_numeric"] = (
        pd.to_numeric(out["temperature"], errors="coerce")
        .fillna(1.0)
        .astype(float)
    )
    out["log_k"] = np.log(out["k"].astype(float))
    return out

fit_model = add_policy_features(
    fit_slopes.merge(
        baseline_features,
        on="query_id",
        how="left",
        validate="many_to_one",
    )
)

val_model = add_policy_features(
    val_slopes.merge(
        baseline_features,
        on="query_id",
        how="left",
        validate="many_to_one",
    )
)

DEPLOYABLE_FEATURES = [
    "pq32_entropy20",
    "pq32_margin1_10",
    "alpha",
    "log_k",
    "is_softmax",
    "temperature_numeric",
]

print("FIT missing deployable cells:", int(fit_model[DEPLOYABLE_FEATURES].isna().sum().sum()))
print("VAL missing deployable cells:", int(val_model[DEPLOYABLE_FEATURES].isna().sum().sum()))

assert fit_model[DEPLOYABLE_FEATURES].notna().all().all()
assert val_model[DEPLOYABLE_FEATURES].notna().all().all()

print("DEPLOYABLE FEATURE TABLE — PASS")


In [ ]:

# ============================================================
# Cell 6 — Fit deployable selector on FIT; evaluate untouched VAL
# ============================================================

X_fit = fit_model[DEPLOYABLE_FEATURES].to_numpy(np.float64)
y_fit = fit_model["is_amplifying_abs"].to_numpy(int)

X_val = val_model[DEPLOYABLE_FEATURES].to_numpy(np.float64)
y_val = val_model["is_amplifying_abs"].to_numpy(int)

clf = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(
        C=0.5,
        max_iter=5000,
        class_weight="balanced",
        random_state=SEED,
    )),
])

clf.fit(X_fit, y_fit)

p_fit = clf.predict_proba(X_fit)[:, 1]
p_val = clf.predict_proba(X_val)[:, 1]

selector_metrics = pd.DataFrame([
    {
        "split": "fit",
        "roc_auc": roc_auc_score(y_fit, p_fit),
        "pr_auc": average_precision_score(y_fit, p_fit),
        "prevalence": float(y_fit.mean()),
        "brier": brier_score_loss(y_fit, p_fit),
    },
    {
        "split": "validation",
        "roc_auc": roc_auc_score(y_val, p_val),
        "pr_auc": average_precision_score(y_val, p_val),
        "prevalence": float(y_val.mean()),
        "brier": brier_score_loss(y_val, p_val),
    },
])

coef_df = pd.DataFrame({
    "feature": DEPLOYABLE_FEATURES,
    "standardized_logistic_coefficient":
        clf.named_steps["model"].coef_[0],
})

print("DEPLOYABLE SELECTOR METRICS")
display(selector_metrics)

print("\nCOEFFICIENTS")
display(
    coef_df.reindex(
        coef_df["standardized_logistic_coefficient"]
        .abs()
        .sort_values(ascending=False)
        .index
    )
)

selector_metrics.to_csv(
    V015_OUT / "v015_deployable_selector_metrics.csv",
    index=False,
)

coef_df.to_csv(
    V015_OUT / "v015_deployable_selector_coefficients.csv",
    index=False,
)


In [ ]:

# ============================================================
# Cell 7 — Query-cluster bootstrap for deployable selector
# ============================================================

val_pred = val_model[["query_id", "is_amplifying_abs"]].copy()
val_pred["probability"] = p_val

groups = {
    qid: g
    for qid, g in val_pred.groupby("query_id", sort=False)
}

qids = np.array(list(groups.keys()), dtype=object)
rng = np.random.default_rng(SEED)

boot_auc = []
boot_ap = []

for _ in range(BOOTSTRAP_REPS):
    sampled = rng.choice(qids, size=len(qids), replace=True)

    ys = []
    ps = []

    for qid in sampled:
        g = groups[qid]
        ys.append(g["is_amplifying_abs"].to_numpy(int))
        ps.append(g["probability"].to_numpy(np.float64))

    yb = np.concatenate(ys)
    pb = np.concatenate(ps)

    if np.unique(yb).size < 2:
        continue

    boot_auc.append(roc_auc_score(yb, pb))
    boot_ap.append(average_precision_score(yb, pb))

selector_bootstrap = pd.DataFrame([
    {
        "metric": "roc_auc",
        "estimate": roc_auc_score(y_val, p_val),
        "ci_low": np.quantile(boot_auc, 0.025),
        "ci_high": np.quantile(boot_auc, 0.975),
        "bootstrap_reps": len(boot_auc),
    },
    {
        "metric": "pr_auc",
        "estimate": average_precision_score(y_val, p_val),
        "ci_low": np.quantile(boot_ap, 0.025),
        "ci_high": np.quantile(boot_ap, 0.975),
        "bootstrap_reps": len(boot_ap),
    },
])

display(selector_bootstrap)

selector_bootstrap.to_csv(
    V015_OUT / "v015_deployable_selector_cluster_bootstrap.csv",
    index=False,
)


In [ ]:

# ============================================================
# Cell 8 — Top-risk enrichment
# ============================================================

ranked = val_model[
    ["query_id", "config_key", "is_amplifying_abs"]
].copy()

ranked["probability"] = p_val

rows = []

for frac in [0.10, 0.25, 0.50]:
    n = max(1, int(math.ceil(len(ranked) * frac)))

    top = ranked.nlargest(n, "probability")

    prevalence = float(ranked["is_amplifying_abs"].mean())
    top_rate = float(top["is_amplifying_abs"].mean())

    rows.append({
        "top_fraction": frac,
        "rows_selected": n,
        "overall_prevalence": prevalence,
        "selected_amplification_rate": top_rate,
        "enrichment_vs_prevalence":
            top_rate / prevalence if prevalence > 0 else np.nan,
    })

enrichment = pd.DataFrame(rows)

display(enrichment)

enrichment.to_csv(
    V015_OUT / "v015_deployable_selector_enrichment.csv",
    index=False,
)



### 25% routing audit

The next analysis evaluates whether selecting the top 25% highest-risk validation query-policy rows catches more actual amplification events than a size-matched random allocation.

This is an **event-capture audit**, not yet a full end-to-end runtime benchmark. It avoids pretending that quality recovery can be computed without a separately defined intervention trajectory.


In [ ]:

# ============================================================
# Cell 9 — 25% risk routing vs matched random
# ============================================================

budget = 0.25
n_select = int(math.ceil(len(ranked) * budget))

selected_idx = ranked.nlargest(n_select, "probability").index
selected_capture = int(ranked.loc[selected_idx, "is_amplifying_abs"].sum())
total_amp = int(ranked["is_amplifying_abs"].sum())

risk_capture_fraction = (
    selected_capture / total_amp
    if total_amp > 0
    else np.nan
)

rng = np.random.default_rng(SEED)

random_capture_fractions = []

all_idx = ranked.index.to_numpy()

for _ in range(RANDOM_BASELINE_REPS):
    idx = rng.choice(all_idx, size=n_select, replace=False)
    captured = int(ranked.loc[idx, "is_amplifying_abs"].sum())

    random_capture_fractions.append(
        captured / total_amp
        if total_amp > 0
        else np.nan
    )

random_capture_fractions = np.asarray(
    random_capture_fractions,
    dtype=np.float64,
)

routing_summary = pd.DataFrame([{
    "budget_fraction": budget,
    "selected_rows": n_select,
    "total_amplifying_rows": total_amp,
    "risk_selected_capture_fraction": risk_capture_fraction,
    "random_mean_capture_fraction":
        float(np.mean(random_capture_fractions)),
    "random_ci_low":
        float(np.quantile(random_capture_fractions, 0.025)),
    "random_ci_high":
        float(np.quantile(random_capture_fractions, 0.975)),
    "risk_minus_random_mean":
        float(
            risk_capture_fraction
            - np.mean(random_capture_fractions)
        ),
    "randomization_p_one_sided":
        float(
            (
                1
                + np.sum(
                    random_capture_fractions
                    >= risk_capture_fraction
                )
            )
            / (len(random_capture_fractions) + 1)
        ),
}])

display(routing_summary)

routing_summary.to_csv(
    V015_OUT / "v015_deployable_selector_25pct_routing.csv",
    index=False,
)



## Part B — Signed harmful-amplification audit

The frozen ARC-v0.13 checkpoints only persist `abs_utility_gap`, not the signed per-round PQ32 and SQ8 utilities.

Therefore the notebook must locate a persisted artifact containing per-query, per-round lower- and higher-fidelity utility columns. It will **fail closed** if such an artifact is not found.

Accepted conceptual forms include columns equivalent to:

- `query_id`
- `iteration`
- `config_key`
- `low_ndcg`
- `high_ndcg`

or explicit PQ32/SQ8 utility columns.

If those values were never persisted, the signed audit requires a **lightweight utility reconstruction from saved retrieved IDs + qrels**, not a full retrieval rerun. This notebook does not silently fabricate signed direction from the absolute gap.


In [ ]:

# ============================================================
# Cell 10 — Search for signed utility trajectory artifacts
# ============================================================

signed_candidates = []

utility_column_sets = [
    {"query_id", "iteration", "config_key", "low_ndcg", "high_ndcg"},
    {"query_id", "iteration", "config_key", "pq32_ndcg", "sq8_ndcg"},
    {"query_id", "iteration", "config_key", "ndcg_low", "ndcg_high"},
]

for p in ARC_ROOT.rglob("*"):
    if not p.is_file():
        continue
    if p.suffix.lower() not in {".parquet", ".csv"}:
        continue

    try:
        if p.suffix.lower() == ".parquet":
            tmp = pd.read_parquet(p)
        else:
            tmp = pd.read_csv(p)

        cols = set(tmp.columns)

        matched_schema = None
        for schema in utility_column_sets:
            if schema.issubset(cols):
                matched_schema = schema
                break

        if matched_schema is not None:
            signed_candidates.append((p, matched_schema, tmp.shape))
            print("SIGNED MATCH:", p, "shape=", tmp.shape, "schema=", matched_schema)

    except Exception:
        pass

if not signed_candidates:
    print()
    print("=" * 80)
    print("SIGNED UTILITY SOURCE NOT FOUND")
    print("=" * 80)
    print(
        "No persisted per-round signed PQ32/SQ8 utility artifact was found.\n"
        "This is not treated as a failure of the scientific result.\n"
        "The signed audit cannot be inferred from abs_utility_gap alone.\n"
        "Export/reconstruct signed utilities from saved retrieved IDs + qrels, "
        "then rerun Cells 10 onward."
    )

SIGNED_UTILITY_AVAILABLE = bool(signed_candidates)

print("SIGNED_UTILITY_AVAILABLE:", SIGNED_UTILITY_AVAILABLE)


In [ ]:

# ============================================================
# Cell 11 — Signed H3 / harmful amplification audit
# Runs only if a signed utility artifact exists.
# ============================================================

if SIGNED_UTILITY_AVAILABLE:

    SIGNED_SOURCE = signed_candidates[0][0]

    if SIGNED_SOURCE.suffix.lower() == ".parquet":
        signed_raw = pd.read_parquet(SIGNED_SOURCE)
    else:
        signed_raw = pd.read_csv(SIGNED_SOURCE)

    cols = set(signed_raw.columns)

    if {"low_ndcg", "high_ndcg"}.issubset(cols):
        low_col, high_col = "low_ndcg", "high_ndcg"

    elif {"pq32_ndcg", "sq8_ndcg"}.issubset(cols):
        low_col, high_col = "pq32_ndcg", "sq8_ndcg"

    elif {"ndcg_low", "ndcg_high"}.issubset(cols):
        low_col, high_col = "ndcg_low", "ndcg_high"

    else:
        raise RuntimeError("Signed utility schema unexpectedly unresolved")

    signed_raw = signed_raw.copy()
    signed_raw["G_signed"] = (
        signed_raw[high_col].astype(float)
        - signed_raw[low_col].astype(float)
    )

    signed_rows = []

    for keys, g in signed_raw.groupby(
        ["query_id", "config_key"],
        sort=False,
    ):
        qid, cfgk = keys
        g = g.sort_values("iteration")
        x = g["iteration"].to_numpy(np.float64)
        y = g["G_signed"].to_numpy(np.float64)

        g0 = float(g.iloc[0]["G_signed"])
        gT = float(g.iloc[-1]["G_signed"])
        signed_slope = float(np.polyfit(x, y, 1)[0])

        signed_rows.append({
            "query_id": qid,
            "config_key": cfgk,
            "G0": g0,
            "GT": gT,
            "delta_G": gT - g0,
            "H3_signed_slope": signed_slope,
        })

    signed_df = pd.DataFrame(signed_rows)

    combined = val_slopes.merge(
        signed_df,
        on=["query_id", "config_key"],
        how="inner",
        validate="one_to_one",
    )

    assert len(combined) > 0

    TIE_EPS = 1e-12

    amp_mask = combined["H3_abs_slope"] > EPS

    combined["signed_taxonomy"] = "not_abs_amplifying"

    combined.loc[
        amp_mask & (combined["GT"] > TIE_EPS),
        "signed_taxonomy",
    ] = "harmful_amplification"

    combined.loc[
        amp_mask & (combined["GT"] < -TIE_EPS),
        "signed_taxonomy",
    ] = "beneficial_divergence"

    combined.loc[
        amp_mask & (np.abs(combined["GT"]) <= TIE_EPS),
        "signed_taxonomy",
    ] = "unresolved_or_tied"

    amp_only = combined.loc[amp_mask].copy()

    taxonomy_summary = (
        amp_only["signed_taxonomy"]
        .value_counts(dropna=False)
        .rename_axis("category")
        .reset_index(name="count")
    )

    taxonomy_summary["fraction"] = (
        taxonomy_summary["count"]
        / taxonomy_summary["count"].sum()
    )

    print("SIGNED TAXONOMY AMONG ABSOLUTE AMPLIFICATION EVENTS")
    display(taxonomy_summary)

    print(
        "P(G_T > 0 | H3_abs > EPS) =",
        float((amp_only["GT"] > TIE_EPS).mean()),
    )

    print(
        "P(H3_signed > 0 | H3_abs > EPS) =",
        float((amp_only["H3_signed_slope"] > 0).mean()),
    )

    combined.to_parquet(
        V015_OUT / "v015_signed_harmful_amplification_rows.parquet",
        index=False,
    )

    taxonomy_summary.to_csv(
        V015_OUT / "v015_signed_harmful_amplification_summary.csv",
        index=False,
    )

else:
    print(
        "Cell 11 skipped because signed per-round utilities were not found."
    )


In [ ]:

# ============================================================
# Cell 12 — Final audit report / seal
# ============================================================

val_selector = selector_metrics.loc[
    selector_metrics["split"] == "validation"
].iloc[0]

report = {
    "status":
        "ARC_V015_DEPLOYABLE_BOUNDARY_SIGNED_HARM_AUDIT_COMPLETE",

    "source_v013_run":
        str(V013_RUN),

    "source_v013_validation_report_sha256":
        sha256_file(VALIDATION_REPORT_PATH),

    "regime_threshold_abs_slope":
        EPS,

    "deployable_selector_features":
        DEPLOYABLE_FEATURES,

    "forbidden_selector_feature_classes": [
        "qrels-derived utility features",
        "SQ8 scores",
        "SQ8 candidate identities",
        "PQ32-SQ8 divergence features",
    ],

    "validation_selector_roc_auc":
        float(val_selector["roc_auc"]),

    "validation_selector_pr_auc":
        float(val_selector["pr_auc"]),

    "validation_selector_prevalence":
        float(val_selector["prevalence"]),

    "routing_25pct_risk_capture_fraction":
        float(
            routing_summary.iloc[0][
                "risk_selected_capture_fraction"
            ]
        ),

    "routing_25pct_random_mean_capture_fraction":
        float(
            routing_summary.iloc[0][
                "random_mean_capture_fraction"
            ]
        ),

    "signed_utility_available":
        bool(SIGNED_UTILITY_AVAILABLE),

    "signed_audit_status":
        (
            "complete"
            if SIGNED_UTILITY_AVAILABLE
            else "blocked_missing_signed_per_round_utility_artifact"
        ),

    "retrieval_rerun":
        False,

    "test_accessed":
        False,

    "interpretation_constraints": [
        (
            "Deployable selector uses only PQ32-side and policy variables; "
            "it does not use qrels or SQ8-side features."
        ),
        (
            "The 25% routing analysis measures amplification-event capture "
            "relative to matched random allocation; it is not itself an "
            "end-to-end quality or latency benchmark."
        ),
        (
            "Signed harmful-amplification conclusions are emitted only if "
            "per-round signed lower/higher utility artifacts are available. "
            "Absolute utility gap alone is insufficient to infer direction."
        ),
    ],

    "completed_at_utc":
        datetime.now(timezone.utc).isoformat(),
}

REPORT_PATH = V015_OUT / "v015_audit_report.json"

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

report_sha = sha256_file(REPORT_PATH)

(V015_OUT / "V015_REPORT_SHA256.txt").write_text(
    report_sha + "  " + REPORT_PATH.name + "\n",
    encoding="utf-8",
)

print()
print("=" * 80)
print("ARC-v0.15 FINAL AUDIT — PASS")
print("=" * 80)
print("Output:", V015_OUT)
print("Report SHA-256:", report_sha)
print()
print(
    "Deployable VAL ROC-AUC:",
    float(val_selector["roc_auc"]),
)
print(
    "Deployable VAL PR-AUC:",
    float(val_selector["pr_auc"]),
)
print(
    "Validation prevalence:",
    float(val_selector["prevalence"]),
)
print(
    "25% risk capture:",
    float(
        routing_summary.iloc[0][
            "risk_selected_capture_fraction"
        ]
    ),
)
print(
    "25% random capture:",
    float(
        routing_summary.iloc[0][
            "random_mean_capture_fraction"
        ]
    ),
)
print(
    "Signed audit:",
    report["signed_audit_status"],
)
print("=" * 80)



## Interpretation rules

### Deployable selector

Do **not** call the selector operationally successful merely because ROC-AUC > 0.5.

A useful result would satisfy at least one of:

- validation ROC-AUC \(\ge 0.65\);
- validation PR-AUC clearly above the amplification prevalence baseline;
- top-25% risk routing captures substantially more amplification events than matched random.

A stronger result would show ROC-AUC \(\ge 0.70\) with stable query-cluster bootstrap confidence intervals.

### Signed audit

The signed audit is a validity audit, not a hypothesis-confirmation exercise.

If:

\[
P(G_T > 0 \mid H3_{abs} > 0.002)
\]

is high, the paper may cautiously describe most absolute-gap amplification events as *harmful to lower fidelity*.

If it is near 0.5, the safer terminology is:

> approximation-induced trajectory divergence

rather than:

> approximation-error amplification

Either outcome is scientifically valid and should be retained.
